# 02b_Copy/Move matching .nxs into triaged HDF folders

This notebook matches raw `.nxs` files to Pixium HDF files and copies/moves them into the same triaged folders for downstream processing.

- Matching is configurable and can strip differing prefixes (e.g. `HDF_` vs `NXS_`).
- Supports dry-run, exact/fuzzy matching, and writes a CSV report.
- Set your paths and prefix rules in the next cell, then run cells in order.

In [ ]:
# Imports
from pathlib import Path
import shutil
import csv
from typing import Dict, List, Optional, Tuple
import pandas as pd

## Configuration
Adjust these values to your environment. If HDF and NXS filenames have different prefixes, list them below to normalize matching.

In [ ]:
# Paths: set your run directory once (contains i11-1-*.nxs files and 02_V7_diff_sorting_output/)
RUN_ROOT = Path(r"E:/I11BT_dec25_dlm_gly/Data_Processing/RAW_2D/Run8_DLM_0.05VF_X2/")
SOURCE_NXS = RUN_ROOT                                # folder containing i11-1-<collection>.nxs
DEST_HDF   = RUN_ROOT / "02_V7_diff_sorting_output"   # contains background/diffraction/maybe folders

# Only scan these triage folders inside DEST_HDF
TRIAGE_FOLDERS = ["background", "diffraction", "maybe"]

# Matching behavior
MODE    = "copy"      # 'copy' or 'move'
FUZZY   = False       # if True, falls back to case-insensitive contains matching
DRY_RUN = False        # if True, no files are modified; only reporting

# Extensions to search
EXT_HDF = ['.h5', '.hdf5', '.hdf']
EXT_NXS = ['.nxs']

# Reporting
REPORT_PATH = Path('nxs_copy_report.csv')  # written in working directory

# Prefix normalization for your naming scheme
HDF_PREFIXES_TO_STRIP = ['pixium_']
NXS_PREFIXES_TO_STRIP = ['i11-1-']

# Optional suffixes to strip (e.g., trailing tokens)
HDF_SUFFIXES_TO_STRIP = []
NXS_SUFFIXES_TO_STRIP = []

# Sanity check (these must exist before running the matching cell)
print('Run root:', RUN_ROOT)
print('NXS source root:', SOURCE_NXS)
print('HDF destination root:', DEST_HDF)
print('Triage folders:', TRIAGE_FOLDERS)

## Utilities
Helper functions for listing files, normalizing stems, indexing and matching.

In [ ]:
def list_files_by_ext(root: Path, exts: List[str]) -> List[Path]:
    exts_lc = {e.lower() for e in exts}
    files: List[Path] = []
    for p in root.rglob('*'):
        if p.is_file() and p.suffix.lower() in exts_lc:
            files.append(p)
    return files

def strip_prefixes(stem: str, prefixes: List[str]) -> str:
    s = stem
    for pref in prefixes:
        if s.startswith(pref):
            s = s[len(pref):]
    return s

def strip_suffixes(stem: str, suffixes: List[str]) -> str:
    s = stem
    for suff in suffixes:
        if s.endswith(suff):
            s = s[: -len(suff)]
    return s

def normalize_stem(stem: str, prefixes: List[str], suffixes: List[str]) -> str:
    s = strip_prefixes(stem, prefixes)
    s = strip_suffixes(s, suffixes)
    return s

def build_index(paths: List[Path], prefixes: List[str], suffixes: List[str]) -> Dict[str, List[Path]]:
    index: Dict[str, List[Path]] = {}
    for p in paths:
        norm = normalize_stem(p.stem, prefixes, suffixes)
        index.setdefault(norm, []).append(p)
    return index

def find_match_for_hdf(hdf: Path, nxs_index: Dict[str, List[Path]], fuzzy: bool, hdf_prefixes: List[str], hdf_suffixes: List[str]) -> Optional[Path]:
    hnorm = normalize_stem(hdf.stem, hdf_prefixes, hdf_suffixes)
    # Exact match first
    if hnorm in nxs_index:
        return nxs_index[hnorm][0]
    if not fuzzy:
        return None
    # Fuzzy: case-insensitive contains
    hnorm_lc = hnorm.lower()
    for nstem, candidates in nxs_index.items():
        nlc = nstem.lower()
        if hnorm_lc in nlc or nlc in hnorm_lc:
            return candidates[0]
    return None

def copy_or_move(src: Path, dst_dir: Path, mode: str) -> Tuple[str, Path]:
    dst_dir.mkdir(parents=True, exist_ok=True)
    dst = dst_dir / src.name
    if mode == 'copy':
        if not dst.exists():
            shutil.copy2(src, dst)
            return ('copied', dst)
        return ('skipped_exists', dst)
    elif mode == 'move':
        if not dst.exists():
            shutil.move(str(src), str(dst))
            return ('moved', dst)
        return ('skipped_exists', dst)
    else:
        raise ValueError(f'Unsupported mode: {mode}')

print('Utilities loaded.')

## Match and (dry) run
Scans HDF triaged folders, finds matching NXS from source, and either plans actions (dry-run) or executes copy/move.

In [ ]:
# Validate paths
assert SOURCE_NXS.exists(), f'Source NXS root does not exist: {SOURCE_NXS}'
assert DEST_HDF.exists(), f'Destination HDF root does not exist: {DEST_HDF}'

# Collect HDF files only from configured triage folders
hdf_files = []
missing_triage = []
for folder_name in TRIAGE_FOLDERS:
    folder_path = DEST_HDF / folder_name
    if folder_path.exists() and folder_path.is_dir():
        hdf_files.extend(list_files_by_ext(folder_path, EXT_HDF))
    else:
        missing_triage.append(folder_path)

# Collect NXS files from source root
nxs_files = list_files_by_ext(SOURCE_NXS, EXT_NXS)

if missing_triage:
    print('Warning: missing triage folders:')
    for p in missing_triage:
        print(' -', p)

print(f'HDF files found (triage folders only): {len(hdf_files)}')
print(f'NXS files found: {len(nxs_files)}')

# Build index on normalized NXS stems
nxs_index = build_index(nxs_files, NXS_PREFIXES_TO_STRIP, NXS_SUFFIXES_TO_STRIP)

rows = []
total_matches = 0
total_actions = 0

for hdf in hdf_files:
    match = find_match_for_hdf(hdf, nxs_index, fuzzy=FUZZY, hdf_prefixes=HDF_PREFIXES_TO_STRIP, hdf_suffixes=HDF_SUFFIXES_TO_STRIP)
    if match is None:
        rows.append({
            'hdf_path': str(hdf),
            'nxs_source': '',
            'action': 'no_match',
            'nxs_destination': '',
            'match_type': 'none',
        })
        continue

    total_matches += 1
    dst_dir = hdf.parent
    if DRY_RUN:
        rows.append({
            'hdf_path': str(hdf),
            'nxs_source': str(match),
            'action': f'planned_{MODE}',
            'nxs_destination': str(dst_dir / match.name),
            'match_type': 'exact' if normalize_stem(hdf.stem, HDF_PREFIXES_TO_STRIP, HDF_SUFFIXES_TO_STRIP) == normalize_stem(match.stem, NXS_PREFIXES_TO_STRIP, NXS_SUFFIXES_TO_STRIP) else 'fuzzy',
        })
    else:
        action, dst = copy_or_move(match, dst_dir, MODE)
        total_actions += 1 if action in ('copied', 'moved') else 0
        rows.append({
            'hdf_path': str(hdf),
            'nxs_source': str(match),
            'action': action,
            'nxs_destination': str(dst),
            'match_type': 'exact' if normalize_stem(hdf.stem, HDF_PREFIXES_TO_STRIP, HDF_SUFFIXES_TO_STRIP) == normalize_stem(match.stem, NXS_PREFIXES_TO_STRIP, NXS_SUFFIXES_TO_STRIP) else 'fuzzy',
        })

print(f'Matches: {total_matches}, actions performed: {total_actions}')
df = pd.DataFrame(rows)
display(df.head(20))

## Write report
Writes the CSV report to `REPORT_PATH`.

In [ ]:
def write_report(rows: List[Dict[str, str]], report_path: Path) -> None:
    fieldnames = ['hdf_path','nxs_source','action','nxs_destination','match_type']
    report_path.parent.mkdir(parents=True, exist_ok=True)
    with report_path.open('w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)

write_report(rows, REPORT_PATH)
print('Report written:', REPORT_PATH)

### Notes
- Set `DRY_RUN = True` to preview actions without modifying files.
- If your stems differ by additional tokens (e.g., timestamps), add them as suffixes in the config.
- For complex renaming rules, we can add a custom mapping function based on regex or tokenization.